Benötigte Module importieren und Datei laden. Die ersten Zeilen werden ausgegeben.

In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

path = "../Data/iris.csv"
data = pd.read_csv(path, delimiter=',')
print(data.head())
print("Empty columns: ", data.columns[data.isnull().any()])

   sepal.length  sepal.width  petal.length  petal.width species
0           5.1          3.5           1.4          0.2  Setosa
1           4.9          3.0           1.4          0.2  Setosa
2           4.7          3.2           1.3          0.2  Setosa
3           4.6          3.1           1.5          0.2  Setosa
4           5.0          3.6           1.4          0.2  Setosa
Empty columns:  Index([], dtype='object')


In [2]:
# Ausgabe der Korrelationen
correlations = data[data.columns].corr(numeric_only=True)
print('All correlations')
print('-' * 30)
correlations_abs_sum = correlations[correlations.columns].abs().sum()
print(correlations_abs_sum)
print('Weakest correlations')
print('-' * 30)
print(correlations_abs_sum.nsmallest(5))

All correlations
------------------------------
sepal.length    2.807265
sepal.width     1.912136
petal.length    3.263059
petal.width     3.146932
dtype: float64
Weakest correlations
------------------------------
sepal.width     1.912136
sepal.length    2.807265
petal.width     3.146932
petal.length    3.263059
dtype: float64


Daten vorbereiten.

In [3]:
# data['sepal.width'] könnte man evtl weglassen, hat geringe Korrelation -> testen
data = data.drop(['sepal.width'], axis = 1)

# Diese Spalte soll vorhergesagt werden 
col = data['sepal.length']
data = data.drop(['sepal.length'], axis = 1)

# führe OHE für diese Daten durch
conv_ohe = ['species']
data = pd.get_dummies(data, columns = conv_ohe, dtype=float)

# Erzeuge Objekt
s_scaler = StandardScaler()
# Spalten für StandardScaler
cols_to_s_scale = ['petal.length', 'petal.width']
data[cols_to_s_scale] = s_scaler.fit_transform(data[cols_to_s_scale])

KNN aufbauen

In [4]:
# Aus den zwei Tabellen vier Tabellen erzeugen
train_data, test_data, train_col, test_col = train_test_split(data,col, test_size=0.2, random_state=42)

# Aufbau KNN
model = tf.keras.Sequential()
model.add(tf.keras.Input(shape=(data.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation=tf.nn.sigmoid))
model.add(tf.keras.layers.Dense(64, activation=tf.nn.sigmoid))
model.add(tf.keras.layers.Dense(1))

# Konfiguration des Lernprozesses
model.compile(optimizer='adam', loss='mae', metrics=['mae'])

Trainieren

In [5]:
# 70 Durchläufe
model.fit(train_data, train_col, epochs=70)

Epoch 1/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 4.4158 - mae: 4.4158 
Epoch 2/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 4.1215 - mae: 4.1215  
Epoch 3/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.8514 - mae: 3.8514 
Epoch 4/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.6190 - mae: 3.6190 
Epoch 5/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.2279 - mae: 3.2279 
Epoch 6/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.9431 - mae: 2.9431 
Epoch 7/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.6661 - mae: 2.6661 
Epoch 8/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.3287 - mae: 2.3287 
Epoch 9/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.0824 - mae: 2.0824 
Epoch 10/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.7613 - mae: 1.7613 
Epoch 11/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4515 - mae: 1.4515 
Epoch 12/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.1063 - mae: 1.1063 
Epoch 13/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.

Testen

In [6]:
test_loss, test_mae = model.evaluate(test_data, test_col)
print('Test mae:', test_mae)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - loss: 0.2951 - mae: 0.2951
Test mae: 0.295141339302063
